# Chapitre 9 - Lab : consensus et coordination avec etcd

etcd est le service de coordination répliqué par **Raft** (cours 5 et 6). Ce lab
lance une grappe de trois nœuds et vérifie sur le vrai système ce que les cours ont
simulé : révisions croissantes et CAS, baux et clés éphémères, verrou dont le bail expire
et jeton de clôture, lectures linéarisables contre lectures sérialisables sur un nœud
isolé, élection d'un nouveau leader après une panne, perte du quorum.

**Démarrer et arrêter la grappe** (depuis la racine du dépôt) :

```bash
docker compose -f docker/consistency/compose.yml -p ddia_consistency up -d
# ... exécuter le notebook ...
docker compose -f docker/consistency/compose.yml -p ddia_consistency down -v
```

Le notebook parle à etcd par sa passerelle JSON/HTTP (ports 11001 à 11003) et pilote Docker
par `subprocess` pour provoquer pannes et coupures réseau (uniquement sur les conteneurs
`ddia_consistency_*`).

**Plan.**

1. La grappe : qui est leader ?
2. Révisions et compare-and-set.
3. Baux et clés éphémères.
4. Un verrou dont le bail expire, et la clôture.
5. Lectures linéarisables et sérialisables sur un nœud isolé.
6. Panne du leader et élection.
7. Perte du quorum.

In [1]:
import base64
import json
import subprocess
import threading
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)

PORTS = {"etcd1": 11001, "etcd2": 11002, "etcd3": 11003}
CONT = {n: f"ddia_consistency_{n}" for n in PORTS}
NET = "ddia_consistency_default"

b64 = lambda s: base64.b64encode(s.encode()).decode()
unb = lambda s: base64.b64decode(s).decode()

def api(node, path, body, timeout=3.0):
    """Appel de la passerelle JSON d'etcd (API v3) sur un nœud."""
    req = urllib.request.Request(f"http://127.0.0.1:{PORTS[node]}/v3/{path}", data=json.dumps(body).encode(), method="POST")
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

def put(node, key, value, lease=None, timeout=3.0):
    body = {"key": b64(key), "value": b64(value)}
    if lease:
        body["lease"] = lease
    return api(node, "kv/put", body, timeout)

def get(node, key, serializable=False, timeout=3.0):
    r = api(node, "kv/range", {"key": b64(key), "serializable": serializable}, timeout)
    kvs = r.get("kvs", [])
    return (unb(kvs[0]["value"]) if kvs else None), r["header"]

def docker(*args):
    return subprocess.run(["docker", *args], capture_output=True, text=True)

def ctl(node, *args):
    """etcdctl exécuté DANS le conteneur (fonctionne même s'il est coupé du réseau)."""
    p = docker("exec", CONT[node], "etcdctl", "--endpoints=127.0.0.1:2379", "--command-timeout=2s", *args)
    return p.returncode, p.stdout.strip(), p.stderr.strip()

def statut():
    rows, ids = [], {}
    for n in PORTS:
        try:
            s = api(n, "maintenance/status", {}, timeout=1.0)
            ids[s["header"]["member_id"]] = n
            rows.append(dict(nœud=n, id=s["header"]["member_id"], leader_id=s.get("leader"), mandat=int(s["raftTerm"]),
                             indice_raft=int(s["raftIndex"]), révision=int(s["header"]["revision"]),
                             erreur=";".join(s.get("errors", []))))
        except Exception as e:
            rows.append(dict(nœud=n, erreur=type(e).__name__))
    df = pd.DataFrame(rows)
    df["leader"] = df.get("leader_id", pd.Series(dtype=object)).map(ids)
    return df.drop(columns=[c for c in ("id", "leader_id") if c in df.columns])

def leader():
    df = statut()
    vals = df["leader"].dropna().unique() if "leader" in df else []
    return vals[0] if len(vals) == 1 else None

def attendre_grappe(timeout=30):
    t0 = time.time()
    while time.time() - t0 < timeout:
        df = statut()
        if "leader" in df and df["leader"].notna().all() and df["leader"].nunique() == 1 and (df["erreur"].fillna("") == "").all():
            return df
        time.sleep(0.3)
    raise RuntimeError("la grappe etcd n'est pas prête : démarrez-la avec docker compose (voir en tête)")

_ = attendre_grappe()